## Test case: Single-thread vs multi-thread

In [ ]:
from pathlib import Path
import time
import os, gzip, tarfile, shutil
from concurrent.futures import ThreadPoolExecutor

raw_tar_path = Path("/Volumes/workspace/weather/weather-data/dataset_DPIRD_raw/dataset_DPIRD_utc0.tar.gz")
untar_folder = Path("dataset_DPIRD_unzip")

# --- First Untar (Sequential) ---
print("--- First Untar (tar.gz) ---")
if untar_folder.exists():
    shutil.rmtree(untar_folder)  # Clean up before test
os.mkdir(untar_folder)

start_time = time.time()
with tarfile.open(raw_tar_path, "r:gz") as tar:
    tar.extractall(path=untar_folder)
dur_first_single = time.time() - start_time

print(f"First-untar time (single-thread): {dur_first_single:.2f} seconds")

# --- Second Untar (Multiple .gz files) ---
print("\n--- Second Untar (.csv.gz) ---")

# Gather all .gz files from a subset of folders for testing
all_dirs = [d for d in untar_folder.rglob('*') if d.is_dir() and len(d.name) == 6 and d.name.isdigit()]
target_files = []
slice_all_dirs = all_dirs[0:9]  # Taking a slice for the test

for folder in slice_all_dirs:
    target_files.extend(list(folder.glob('*.gz')))

"""Helper to unzip a .gz file to .csv"""
def unzip_gz_file(file_path):
    dest_path = file_path.with_suffix('') # remove .gz
    with gzip.open(file_path, 'rb') as f_in, open(dest_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    return dest_path

# 1. Single-thread test
start_time = time.time()
for f in target_files:
    unzip_gz_file(f)
dur_second_single = time.time() - start_time
print(f"Second-untar time (single-thread): {dur_second_single:.2f} seconds")

# Clean up extracted csvs to reset for the multi-thread test
for f in target_files:
    csv_path = f.with_suffix('')
    if csv_path.exists():
        csv_path.unlink()

# 2. Multi-thread test
start_time = time.time()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    list(executor.map(unzip_gz_file, target_files))
dur_second_multi = time.time() - start_time
print(f"Second-untar time (multi-thread): {dur_second_multi:.2f} seconds")

Conclusion: 
<br>
First untar: 88s
<br>
Second-untar (single): 1.08
<br>
Second-untar (multi): 0.10